# 07 · Modelos de sentimiento

**Objetivo:** comparar tres formas de conseguir un modelo que mida el sentimiento de un texto financiero, de no entrenar nada a preentrenar y ajustar: **M1**, el diccionario Loughran-McDonald; **M2**, un Transformer entrenado solo con las frases etiquetadas de **Financial PhraseBank**; y **M3**, la misma red preentrenada como BERT sobre el corpus propio y ajustada después con PhraseBank. Todos usan la misma partición y las mismas métricas, para que la comparación sea justa. M2 y M3 siguen el código de los apuntes de clase (`Transformers.ipynb` y `bert_simplificado.ipynb`).

**Plan:**

| Paso | Qué se hace | Estado |
|---|---|---|
| 1 | Financial PhraseBank: lectura, clases, longitudes, repetidas y solapamiento con el corpus | hecho |
| 2 | Partición estratificada 60/20/20 sin repetidas; referencia "siempre neutral" | hecho |
| 3 | Función de evaluación (en `funciones.py`, la usa también el 08): F1 macro, exactitud, matriz de confusión e intervalo de confianza del 95 % por bootstrap | hecho |
| 4 | **M1**, diccionario Loughran-McDonald: fuente, formato y licencia antes de descargarlo; el umbral de "neutral" se elige en validación | hecho |
| 5 | Vocabulario común: `TextVectorization` por palabras con el token `[mask]`, aprendido con el corpus y el entrenamiento de PhraseBank; frases de hasta 64 palabras | hecho |
| 6 | **M2**: `PositionalEmbedding` + `TransformerEncoder` a 3 clases; épocas y pesos de clase elegidos en validación | hecho |
| 7 | **M3**, preentrenamiento: la misma red adivina palabras tapadas (enmascarado de `bert_simplificado`) en el corpus entero, 2 épocas | hecho |
| 8 | **M3**, ajuste con PhraseBank: primero con la red congelada y después descongelada, como en clase | en curso |
| 9 | Tabla final: el test se mira una sola vez, con los tres modelos y la referencia; sensibilidad con AllAgree | pendiente |

Tamaño de M2 y M3, el de `bert_simplificado`: 128 dimensiones, 8 cabezas de atención y 128 neuronas en la capa densa.

**Entradas:**
- Financial PhraseBank (Malo et al., 2014), descargado en `data/phrasebank/`
- `data/cache/corpus_preentrenamiento.parquet` (notebook 06): texto sin etiquetas para preentrenar M3

**Salidas:**
- `data/cache/phrasebank_particion.parquet`: las frases de 50Agree sin repetidas, con su parte (entrenamiento, validación o test) y si están en AllAgree
- `data/modelos/`: `vocabulario.txt`; `m1_diccionario.json`; `m2_sin_pesos.keras` y `m2_con_pesos.keras` con sus historiales y `m2_eleccion.json`; `m3_preentrenado.keras` y su historial
- *(previstas)* en `data/modelos/`: `m3_congelado.keras` y `m3_descongelado.keras` (M3 ajustado) con sus historiales; en `data/cache/`: las predicciones de cada modelo sobre validación y test

In [1]:
import json
import os
import re
import time
import warnings
import zipfile

os.environ["KERAS_BACKEND"] = "tensorflow"   # motor de Keras del proyecto; se fija antes de importar keras

import keras
import numpy as np
import pandas as pd
import tensorflow as tf
from keras import layers
from keras.layers import TextVectorization
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
import plotly.express as px
from dotenv import load_dotenv

from funciones import (
    CARPETA_CACHE,
    CLASES_SENTIMIENTO,
    SEMILLA,
    clasificar_tono,
    comprobar_descarga_manual,
    evaluar_clasificacion,
    tono_loughran_mcdonald,
)

load_dotenv()
print("Keras", keras.__version__, "· motor:", keras.backend.backend())

Keras 3.15.1 · motor: tensorflow


## 1. ¿Con qué frases etiquetadas aprenden los modelos? (Financial PhraseBank)

**Financial PhraseBank**, [arXiv 1307.5336](https://arxiv.org/abs/1307.5336)) son ~4.800 frases de noticias financieras en inglés etiquetadas cada una como **negativa, neutral o positiva**.

- **Fuente:** el zip original de los autores, `FinancialPhraseBank-v1.0.zip`, tal como está en la página del dataset en Hugging Face, [takala/financial_phrasebank](https://huggingface.co/datasets/takala/financial_phrasebank), carpeta `data` ([enlace directo al zip](https://huggingface.co/datasets/takala/financial_phrasebank/resolve/main/data/FinancialPhraseBank-v1.0.zip)). Se descargó a mano el 21 oct 2026 y se guardó en `data/phrasebank/`.
- **Licencia:** CC BY-NC-SA 3.0. Uso académico sin fines comerciales, citando el artículo. `data/` no se sube a GitHub, así que el dataset no se redistribuye.

El notebook comprueba que el zip está y que su **huella SHA-256** es la del fichero descargado: si fuera otra versión o una descarga incompleta, se para en lugar de seguir con otros datos. Se descomprimen solo los ficheros de los autores (el zip trae además carpetas ocultas de macOS).

In [2]:
CARPETA_PHRASEBANK = CARPETA_CACHE.parent / "phrasebank"
RUTA_ZIP = CARPETA_PHRASEBANK / "FinancialPhraseBank-v1.0.zip"
comprobar_descarga_manual(RUTA_ZIP,
                          sha256_esperado="0e1a06c4900fdae46091d031068601e3773ba067c7cecb5b0da1dcba5ce989a6",
                          enlace_descarga="https://huggingface.co/datasets/takala/financial_phrasebank/resolve/main/data/FinancialPhraseBank-v1.0.zip")

with zipfile.ZipFile(RUTA_ZIP) as comprimido:
    ficheros = [nombre for nombre in comprimido.namelist() if nombre.startswith("FinancialPhraseBank-v1.0/") and nombre.endswith(".txt")]
    comprimido.extractall(CARPETA_PHRASEBANK, members=ficheros)
print("Ficheros:", [nombre.split("/")[1] for nombre in ficheros])

Ficheros: ['License.txt', 'README.txt', 'Sentences_50Agree.txt', 'Sentences_66Agree.txt', 'Sentences_75Agree.txt', 'Sentences_AllAgree.txt']


### 1.1 Cómo son los ficheros por dentro

Una frase por línea, con la etiqueta al final tras una `@`. La codificación es **latin-1**, no UTF-8: leído como UTF-8, el fichero falla en el byte 2.566. Así son las tres primeras líneas:

In [3]:
RUTA_50AGREE = CARPETA_PHRASEBANK / "FinancialPhraseBank-v1.0" / "Sentences_50Agree.txt"
for linea in RUTA_50AGREE.read_text(encoding="latin-1").splitlines()[:3]:
    print(linea)

According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing .@neutral
Technopolis plans to develop in stages an area of no less than 100,000 square meters in order to host companies working in computer technologies and telecommunications , the statement said .@neutral
The international electronic industry company Elcoteq has laid off tens of employees from its Tallinn facility ; contrary to earlier layoffs the company contracted the ranks of its office workers , the daily Postimees reported .@negative


Para leer cada variante se separa cada línea por la **última** `@` (por si alguna frase tuviera otra). Fíjate en que las frases traen **espacios antes de la puntuación** (*"Gran , the company"*): es como las dejaron los autores.

In [4]:
def leer_phrasebank(variante: str) -> pd.DataFrame:
    """Lee una variante de PhraseBank ('50Agree', 'AllAgree'…) y devuelve una tabla con frase y etiqueta."""
    ruta = CARPETA_PHRASEBANK / "FinancialPhraseBank-v1.0" / f"Sentences_{variante}.txt"
    filas = []
    for linea in ruta.read_text(encoding="latin-1").splitlines():
        frase, etiqueta = linea.rsplit("@", 1)
        filas.append({"frase": frase.strip(), "etiqueta": etiqueta.strip()})
    return pd.DataFrame(filas)


frases_50 = leer_phrasebank("50Agree")
frases_todas = leer_phrasebank("AllAgree")
print(f"50Agree: {len(frases_50):,} frases · AllAgree: {len(frases_todas):,} frases")

50Agree: 4,846 frases · AllAgree: 2,264 frases


### 1.2 ¿Cuántas frases hay de cada clase?

In [5]:
distribucion = pd.DataFrame({
    "50Agree": frases_50["etiqueta"].value_counts(),
    "50Agree (%)": (frases_50["etiqueta"].value_counts(normalize=True) * 100).round(1),
    "AllAgree": frases_todas["etiqueta"].value_counts(),
    "AllAgree (%)": (frases_todas["etiqueta"].value_counts(normalize=True) * 100).round(1),
})
distribucion

,50Agree,50Agree (%),AllAgree,AllAgree (%)
etiqueta,,,,
neutral,2879,59.4,1391,61.4
positive,1363,28.1,570,25.2
negative,604,12.5,303,13.4


### 1.3 ¿Cómo de largas son?

Longitud en palabras (contando como palabra cada trozo separado por espacios, signos de puntuación incluidos).

In [6]:
frases_50["palabras"] = frases_50["frase"].str.split().str.len()
print(frases_50.groupby("etiqueta")["palabras"].describe(percentiles=[0.5]).round(0)[["count", "min", "50%", "max"]])

grafico = px.histogram(frases_50, x="palabras", color="etiqueta", nbins=40, barmode="overlay",
                       title="Longitud de las frases de 50Agree, en palabras")
grafico.show(renderer="notebook_connected")

           count  min   50%   max
etiqueta                         
negative   604.0  5.0  22.0  56.0
neutral   2879.0  2.0  21.0  81.0
positive  1363.0  5.0  23.0  57.0


### 1.4 Ejemplos de cada clase

In [7]:
pd.set_option("display.max_colwidth", 160)
frases_50.groupby("etiqueta").sample(3, random_state=SEMILLA)[["etiqueta", "frase"]]

,etiqueta,frase
3790,negative,The company decided at the end of 2008 to temporarily shut down its ammonia plant in Billingham and extend the maintenance period at its Ince facility .
4670,negative,down to EUR5 .9 m H1 '09 3 August 2009 - Finnish media group Ilkka-Yhtyma Oyj ( HEL : ILK2S ) said today its net profit fell 45 % on the year to EUR5 .9 m i...
4797,negative,"The steelmaker said that the drop in profit was explained by the continuing economic uncertainty , mixed with the current drought in bank lending , resultin..."
1636,neutral,The mill is concentrating on getting the supercalendered line running satisfactorily before restarting its older newsprint line .
1117,neutral,Wellness services will be important in the future .
2775,neutral,"After completion of the acquisition , Poyry 's oil and gas expertise will cover multi-discipline engineering and consulting services across the complete oil..."
2310,positive,The fund at fair value will increase correspondingly .
2317,positive,This new partnership agreement represents a significant milestone for both parties .
1479,positive,`` The CHF is a great product .


### 1.5 ¿Hay frases repetidas?

Una frase repetida con **la misma** etiqueta solo pesa doble. Repetida con etiquetas **distintas** es peor: el modelo recibe dos respuestas contradictorias para la misma pregunta.

In [8]:
for nombre, tabla in [("50Agree", frases_50), ("AllAgree", frases_todas)]:
    repetidas = tabla[tabla.duplicated(subset="frase", keep=False)]
    etiquetas_por_frase = repetidas.groupby("frase")["etiqueta"].nunique()
    print(f"{nombre}: {repetidas['frase'].nunique()} frases repetidas ({len(repetidas)} filas)"
          f" · con etiquetas distintas: {(etiquetas_por_frase > 1).sum()}")

con_conflicto = frases_50.groupby("frase")["etiqueta"].nunique()
frases_50[frases_50["frase"].isin(con_conflicto[con_conflicto > 1].index)].sort_values("frase")

50Agree: 8 frases repetidas (16 filas) · con etiquetas distintas: 2
AllAgree: 5 frases repetidas (10 filas) · con etiquetas distintas: 0


,frase,etiqueta,palabras
78,TELECOMWORLDWIRE-7 April 2006-TJ Group Plc sells stake in Morning Digital Design Oy Finnish IT company TJ Group Plc said on Friday 7 April that it had signe...,neutral,42
79,TELECOMWORLDWIRE-7 April 2006-TJ Group Plc sells stake in Morning Digital Design Oy Finnish IT company TJ Group Plc said on Friday 7 April that it had signe...,positive,42
788,The Group 's business is balanced by its broad portfolio of sports and presence in all major markets .,positive,19
789,The Group 's business is balanced by its broad portfolio of sports and presence in all major markets .,neutral,19


### 1.6 ¿Aparece alguna frase de PhraseBank en nuestro corpus?

Si una frase del test estuviera también en el corpus de preentrenamiento del notebook 06, el BERT y el GPT la habrían "leído" antes de examinarse de ella. Se compara el texto **normalizado**: en minúsculas y **sin ningún espacio**, porque PhraseBank pone espacios antes de la puntuación (*"Gran , the"*) y el corpus no (*"Gran, the"*).

In [9]:
def normalizar(texto: str) -> str:
    """Texto en minúsculas y sin espacios, para comparar frases escritas con distinto espaciado."""
    return re.sub(r"\s+", "", texto.lower())


corpus = pd.read_parquet(CARPETA_CACHE / "corpus_preentrenamiento.parquet", columns=["texto", "fuente"])
corpus_normalizado = set(corpus["texto"].map(normalizar))
en_el_corpus = frases_50["frase"].map(normalizar).isin(corpus_normalizado)
print(f"Frases de PhraseBank (50Agree) que aparecen en el corpus: {en_el_corpus.sum()} de {len(frases_50):,}")

Frases de PhraseBank (50Agree) que aparecen en el corpus: 0 de 4,846


**Conclusión:**

- **4.846 frases en 50Agree y 2.264 en AllAgree**, muy desequilibradas: casi 6 de cada 10 son **neutrales** y solo 1 de cada 8 **negativa**. Un modelo que dijera siempre "neutral" acertaría el 59 % sin aprender nada; por eso la métrica principal será el F1 macro, que da el mismo peso a las tres clases.
- Son frases **cortas**: unas 22 palabras de mediana y como mucho 81. Para los Transformers bastará con un máximo de 128 tokens.
- **8 frases repetidas en 50Agree**, y **2 de ellas con etiquetas contradictorias** (la misma frase, una vez neutral y otra positiva). En AllAgree hay 5 repetidas, todas con la misma etiqueta.
- **Ninguna frase de PhraseBank aparece en nuestro corpus** de preentrenamiento: el preentrenamiento no habrá visto las frases del test. La comprobación busca frases idénticas (salvo espacios y mayúsculas), no parecidas; es suficiente porque PhraseBank son noticias de empresas finlandesas, casi todas de 2004–2011 (son los años que más citan sus frases), y el corpus empieza en octubre de 2016.

## 2. ¿Con qué frases se entrena, se ajusta y se examina cada modelo? (partición)

### 2.1 Quitar las frases repetidas

Si una frase repetida cae una vez en entrenamiento y otra en test, el modelo se examina de algo que ya ha visto. Antes de partir:

- Las **2 frases con etiquetas contradictorias** se eliminan del todo: no sabemos cuál es la etiqueta buena.
- De las **demás repetidas** se deja una sola copia.

In [10]:
etiquetas_por_frase = frases_50.groupby("frase")["etiqueta"].nunique()
contradictorias = etiquetas_por_frase[etiquetas_por_frase > 1].index

frases = frases_50[~frases_50["frase"].isin(contradictorias)].drop_duplicates(subset="frase").reset_index(drop=True)
frases["en_allagree"] = frases["frase"].isin(frases_todas["frase"])
print(f"50Agree: {len(frases_50):,} filas → sin contradictorias ni repetidas: {len(frases):,} frases"
      f" ({len(frases_50) - len(frases)} filas fuera) · en AllAgree: {frases['en_allagree'].sum():,}")

50Agree: 4,846 filas → sin contradictorias ni repetidas: 4,836 frases (10 filas fuera) · en AllAgree: 2,259


### 2.2 Partición estratificada 60 / 20 / 20

Cada frase va a una de tres partes, al azar pero **manteniendo en cada parte la misma proporción de clases** (estratificada) y con la semilla fija:

| Parte | Para qué se usa |
|---|---|
| **Entrenamiento** (60 %) | El modelo aprende con estas frases |
| **Validación** (20 %) | Elegir hiperparámetros y decidir cuándo parar de entrenar |
| **Test** (20 %) | Examen final: **no se mira hasta la tabla final de todos los modelos (paso 9)** |

Los apuntes de clase usan el test como validación. Eso es una fuga de información: si se eligen las épocas mirando el test, el test deja de ser un examen limpio. Aquí no se hace.

Se parte en dos pasos: primero se separa el 20 % de test, y del 80 % restante se separa una cuarta parte (el 20 % del total) como validación.

In [11]:
resto, test = train_test_split(frases, test_size=0.2, stratify=frases["etiqueta"], random_state=SEMILLA)
entrenamiento, validacion = train_test_split(resto, test_size=0.25, stratify=resto["etiqueta"], random_state=SEMILLA)

particion = pd.concat([entrenamiento.assign(parte="entrenamiento"), validacion.assign(parte="validacion"),
                       test.assign(parte="test")], ignore_index=True)[["frase", "etiqueta", "parte", "en_allagree"]]
particion.to_parquet(CARPETA_CACHE / "phrasebank_particion.parquet", index=False)
print("Frases por parte:", particion["parte"].value_counts().to_dict())

Frases por parte: {'entrenamiento': 2901, 'test': 968, 'validacion': 967}


¿Tiene cada parte la misma proporción de clases? Y, para la medida de sensibilidad, ¿cuántas frases del test están también en AllAgree (todos los anotadores de acuerdo)?

In [12]:
proporciones = (pd.crosstab(particion["parte"], particion["etiqueta"], normalize="index") * 100).round(1)
proporciones["frases"] = particion["parte"].value_counts()
display(proporciones.loc[["entrenamiento", "validacion", "test"]])

test_allagree = particion[(particion["parte"] == "test") & particion["en_allagree"]]
print(f"Test: {(particion['parte'] == 'test').sum()} frases, de las que {len(test_allagree)} están en AllAgree."
      f" Se medirán aparte como sensibilidad: el mismo examen, solo con las frases en las que todos los anotadores coincidieron.")

etiqueta,negative,neutral,positive,frases
parte,,,,
entrenamiento,12.5,59.4,28.2,2901
validacion,12.5,59.4,28.1,967
test,12.5,59.4,28.1,968


Test: 968 frases, de las que 448 están en AllAgree. Se medirán aparte como sensibilidad: el mismo examen, solo con las frases en las que todos los anotadores coincidieron.


### 2.3 La referencia mínima: decir siempre "neutral"

Cualquier modelo tiene que superar a uno que no aprende nada y responde siempre la clase más frecuente. Se calcula sobre la **validación**, para no mirar el test hasta el final.

In [13]:
reales = particion.loc[particion["parte"] == "validacion", "etiqueta"]
siempre_neutral = ["neutral"] * len(reales)
print(f"Siempre neutral (validación) · exactitud: {accuracy_score(reales, siempre_neutral):.3f}"
      f" · F1 macro: {f1_score(reales, siempre_neutral, average='macro'):.3f}")

Siempre neutral (validación) · exactitud: 0.594 · F1 macro: 0.248


**La exactitud sale alta y el F1 macro muy bajo** porque la exactitud solo cuenta aciertos, y acertar todas las neutrales ya es el 59 %. El F1 macro hace la media de las tres clases con el mismo peso, y en negativa y positiva este "modelo" saca un 0.

**A tener en cuenta en M2 y M3 (sin decidir todavía):** con este desequilibrio, una red puede aprender a decir casi siempre "neutral" y conseguir buena exactitud. Una forma habitual de evitarlo son los **pesos de clase**: equivocarse en una negativa cuenta más que en una neutral. Se decidirá al entrenar M2, viendo los resultados en la validación.

**Conclusión:** quedan 4.836 frases repartidas 60/20/20 con la misma proporción de clases en cada parte, guardadas para que todos los modelos usen exactamente las mismas. El listón mínimo es un F1 macro de 0,25.

## 3. ¿Cómo se mide cada modelo? (función de evaluación)

Todos los modelos se miden con la misma función, `evaluar_clasificacion` (`funciones.py`), que también usará el notebook 08 con los titulares etiquetados a mano. Recibe las etiquetas reales y las predichas y devuelve:

| Medida | Qué es |
|---|---|
| **F1 macro** (principal) | El F1 de cada clase (media armónica de precisión y exhaustividad), promediado con el mismo peso para las tres. Una clase pequeña, como la negativa, cuenta lo mismo que la neutral |
| **Intervalo de confianza del 95 %** del F1 macro | Cuánto podría cambiar el F1 con otras frases parecidas. Se calcula por **bootstrap**: se sacan 1.000 muestras al azar **con reemplazamiento** de las mismas frases (cada muestra tiene tantas frases como la original, con unas repetidas y otras fuera), se calcula el F1 en cada una y se toman los percentiles 2,5 y 97,5 |
| **Exactitud** | Porcentaje de frases acertadas |
| **Matriz de confusión** | Cuántas frases de cada clase real acaban en cada clase predicha |

El intervalo sirve para comparar modelos con honestidad: si los intervalos de dos modelos se solapan mucho, la diferencia puede ser casualidad de qué frases tocaron en la validación o el test.

Se prueba con la referencia del paso 2.3, "siempre neutral" sobre la validación, que debe dar el mismo F1 macro (0,248) y la misma exactitud (0,594).

In [14]:
evaluacion_neutral = evaluar_clasificacion(reales, siempre_neutral)

print(f"Siempre neutral (validación, {evaluacion_neutral['frases']} frases)")
print(f"F1 macro: {evaluacion_neutral['f1_macro']:.3f}  (IC 95 %: {evaluacion_neutral['f1_ic_inferior']:.3f} – {evaluacion_neutral['f1_ic_superior']:.3f})")
print(f"Exactitud: {evaluacion_neutral['exactitud']:.3f}")
evaluacion_neutral["matriz_confusion"]

Siempre neutral (validación, 967 frases)
F1 macro: 0.248  (IC 95 %: 0.240 – 0.257)
Exactitud: 0.594


predicha,negative,neutral,positive
real,,,
negative,0,121,0
neutral,0,574,0
positive,0,272,0


La matriz muestra lo esperado: todas las frases, sean de la clase que sean, caen en la columna "neutral".

El intervalo es **estrecho** (unas 0,24 – 0,26) porque este "modelo" no puede acertar ni fallar de formas distintas: su F1 solo depende de qué proporción de neutrales toque en cada remuestreo. En un modelo de verdad el intervalo será más ancho, sobre todo por la clase negativa, que en la validación tiene solo unas 120 frases.

**Conclusión:** la función reproduce las cifras de la referencia. A partir de aquí, cada modelo se resume con su F1 macro y su intervalo, su exactitud y su matriz de confusión.

## 4. M1: ¿cuánto se consigue sin entrenar nada? (diccionario Loughran-McDonald)

### 4.1 El diccionario

El **diccionario Loughran-McDonald** (Loughran y McDonald, 2011, *"When Is a Liability Not a Liability? Textual Analysis, Dictionaries, and 10-Ks"*, Journal of Finance) es la lista de palabras de sentimiento más usada en finanzas. Los autores la construyeron leyendo informes 10-K, porque las listas generales fallaban en este lenguaje: *liability* ("pasivo") o *tax* no son negativas en un informe financiero.

- **Fuente:** la página oficial de los autores en el Software Repository for Accounting and Finance de la Universidad de Notre Dame ([sraf.nd.edu](https://sraf.nd.edu/loughranmcdonald-master-dictionary/)). Desde ella se descargó a mano, el 1 nov 2026, el fichero `Loughran-McDonald_MasterDictionary_1993-2025.csv` (versión de marzo de 2026), con el enlace *CSV* de la página ([enlace directo en Google Drive](https://drive.google.com/file/d/1iq2RUf8qGFEAk1g8wQntP3habOnR3fXF/view)), y se guardó en `data/loughran_mcdonald/`.
- **Licencia:** *"The dictionary/sentiment lists are free for use in academic research"*. Uso académico citando el artículo de 2011. `data/` no se sube a GitHub, así que el diccionario no se redistribuye.

El notebook comprueba que el fichero está y que su **huella SHA-256** es la del fichero descargado: si alguien usara otra versión del diccionario, el notebook se para en lugar de dar otras cifras.

In [15]:
RUTA_LM = CARPETA_CACHE.parent / "loughran_mcdonald" / "Loughran-McDonald_MasterDictionary_1993-2025.csv"
comprobar_descarga_manual(RUTA_LM,
                          sha256_esperado="e2d1328682bab7d2187684fb9f5420bb730401c9eefc00daf835edd203f4859d",
                          enlace_descarga="https://sraf.nd.edu/loughranmcdonald-master-dictionary/ (enlace CSV)")
print(f"{RUTA_LM.name}: {RUTA_LM.stat().st_size / 1e6:.1f} MB")

Loughran-McDonald_MasterDictionary_1993-2025.csv: 9.1 MB


### 4.2 Cómo es el fichero por dentro

Una fila por palabra, en mayúsculas, con sus frecuencias en los 10-K y una columna por cada una de las siete listas: negativa, positiva, incertidumbre, litigiosa, modal fuerte, modal débil y restrictiva.

Se lee con `keep_default_na=False`: si no, pandas convertiría en un dato vacío la palabra *NULL*, que está en el diccionario.

In [16]:
diccionario_lm = pd.read_csv(RUTA_LM, keep_default_na=False)
print("Filas:", len(diccionario_lm), "· columnas:", list(diccionario_lm.columns))
diccionario_lm.head(3)

Filas: 86553 · columnas: ['Word', 'Seq_num', 'Word Count', 'Word Proportion', 'Average Proportion', 'Std Dev', 'Doc Count', 'Negative', 'Positive', 'Uncertainty', 'Litigious', 'Strong_Modal', 'Weak_Modal', 'Constraining', 'Complexity', 'Syllables', 'Source']


,Word,Seq_num,Word Count,Word Proportion,Average Proportion,Std Dev,Doc Count,Negative,Positive,Uncertainty,Litigious,Strong_Modal,Weak_Modal,Constraining,Complexity,Syllables,Source
0,AARDVARK,1,814,3.085383e-08,2.022181e-08,4.059339e-06,158,0,0,0,0,0,0,0,0,2,12of12inf
1,AARDVARKS,2,3,1.137119e-10,7.898767e-12,8.829342e-09,1,0,0,0,0,0,0,0,0,2,12of12inf
2,ABACI,3,9,3.411357e-10,1.067549e-10,5.054031e-08,7,0,0,0,0,0,0,0,0,3,12of12inf


Cada columna de lista no dice "sí" o "no", sino **el año en que la palabra entró** en la lista (0 si nunca estuvo). Un año **con signo menos** indica el año en que se **quitó**. Por eso una palabra pertenece a la lista si el valor es **mayor que 0**, no si es distinto de 0:

In [17]:
LISTAS_LM = ["Negative", "Positive", "Uncertainty", "Litigious", "Strong_Modal", "Weak_Modal", "Constraining"]
pd.DataFrame({
    "en vigor (> 0)": (diccionario_lm[LISTAS_LM] > 0).sum(),
    "quitadas (< 0)": (diccionario_lm[LISTAS_LM] < 0).sum(),
})

,en vigor (> 0),quitadas (< 0)
Negative,2345,10
Positive,347,7
Uncertainty,297,0
Litigious,903,2
Strong_Modal,19,0
Weak_Modal,27,0
Constraining,184,0


In [18]:
print("Quitadas de la lista negativa:", diccionario_lm.loc[diccionario_lm["Negative"] < 0, "Word"].tolist())
print("Quitadas de la lista positiva:", diccionario_lm.loc[diccionario_lm["Positive"] < 0, "Word"].tolist())

Quitadas de la lista negativa: ['BREAKING', 'BRIDGE', 'BROKEN', 'CLOSED', 'CLOSING', 'CRITICAL', 'FORCE', 'FUGITIVE', 'LATE', 'SECRECY']
Quitadas de la lista positiva: ['BENEFICIAL', 'BENEFIT', 'EFFECTIVE', 'GREAT', 'GREATER', 'HONORABLE', 'REWARDS']


M1 usa solo las dos listas de tono, **negativa y positiva**. Las otras cinco (incertidumbre, litigio, modales y restricción) no corresponden a ninguna de las tres clases de PhraseBank. Unos ejemplos al azar de cada una:

In [19]:
palabras_negativas = set(diccionario_lm.loc[diccionario_lm["Negative"] > 0, "Word"])
palabras_positivas = set(diccionario_lm.loc[diccionario_lm["Positive"] > 0, "Word"])
print(f"Negativas: {len(palabras_negativas)} · positivas: {len(palabras_positivas)} · en las dos: {len(palabras_negativas & palabras_positivas)}")
print("Negativas, al azar:", sorted(pd.Series(sorted(palabras_negativas)).sample(12, random_state=SEMILLA)))
print("Positivas, al azar:", sorted(pd.Series(sorted(palabras_positivas)).sample(12, random_state=SEMILLA)))

Negativas: 2345 · positivas: 347 · en las dos: 0
Negativas, al azar: ['ANOMALIES', 'CONVICT', 'DISAVOWED', 'DISTRACTS', 'GRATUITOUS', 'GROSSLY', 'INVESTIGATIONS', 'MALICE', 'OVERTURNS', 'RETALIATION', 'STRAIN', 'TOLERATING']
Positivas, al azar: ['CONCLUSIVELY', 'EMPOWERING', 'ENHANCE', 'EXCLUSIVES', 'IMPRESSES', 'INNOVATION', 'LOYAL', 'PROFICIENCY', 'PROSPEROUS', 'STRENGTH', 'SUCCESSFUL', 'SURPASS']


**Conclusión:** 86.553 palabras, de las que 2.345 son negativas y 347 positivas (casi 7 negativas por cada positiva: los 10-K hablan mucho más de riesgos que de logros). Ninguna palabra está en las dos listas. Dos limitaciones a tener en cuenta al leer los resultados de M1: el diccionario se hizo con informes 10-K, no con titulares, y cuenta palabras sueltas sin tener en cuenta las negaciones (*"not profitable"* suma una palabra positiva).

### 4.3 La regla de M1: del recuento de palabras a una clase

Para cada frase se calcula su **tono**:

$$\text{tono} = \frac{\text{palabras positivas} - \text{palabras negativas}}{\text{palabras de la frase}}$$

Las palabras son los trozos de solo letras de la frase, en mayúsculas, como están en el diccionario (los números y los signos no cuentan). El tono va de −1 a 1 y vale 0 si la frase no tiene ninguna palabra de las listas o si se compensan.

La clase sale de comparar el tono con un **umbral**: *positiva* si el tono es mayor que el umbral, *negativa* si es menor que −umbral y *neutral* en otro caso. Las dos funciones (`tono_loughran_mcdonald` y `clasificar_tono`) están en `funciones.py`, porque el notebook 08 las aplicará a los titulares.

Unos ejemplos de la validación:

In [20]:
validacion = particion[particion["parte"] == "validacion"].copy()
validacion["tono_m1"] = [tono_loughran_mcdonald(frase, palabras_negativas, palabras_positivas) for frase in validacion["frase"]]

pd.set_option("display.max_colwidth", 120)
pd.concat([validacion.nlargest(2, "tono_m1"), validacion.nsmallest(2, "tono_m1")])[["frase", "etiqueta", "tono_m1"]].round(3)

,frase,etiqueta,tono_m1
2985,"`` Our decision and resolve on MeeGo is only stronger , '' she continued .",positive,0.182
3533,Finnair was able to operate most of its leisure flights despite the strike .,positive,0.154
3393,Terms were not disclosed .,neutral,-0.250
3585,No financial details were disclosed .,neutral,-0.200


¿Cuántas frases tienen alguna palabra con tono, según su clase real? Si una frase no tiene ninguna, M1 solo puede decir "neutral":

In [21]:
cobertura = validacion.groupby("etiqueta")["tono_m1"].agg(
    tono_positivo=lambda tono: (tono > 0).mean(),
    tono_negativo=lambda tono: (tono < 0).mean(),
    sin_tono=lambda tono: (tono == 0).mean(),
)
(cobertura * 100).round(1)

,tono_positivo,tono_negativo,sin_tono
etiqueta,,,
negative,1.7,43.0,55.4
neutral,7.1,8.0,84.8
positive,24.6,7.4,68.0


**El umbral se elige en la validación**, probando de 0 a 0,20 en pasos de 0,01 y quedándose con el que da mayor F1 macro (si empatan, el más pequeño). M1 no aprende nada más: es la única decisión que se toma con datos.

In [22]:
umbrales = np.round(np.arange(0, 0.201, 0.01), 2)
f1_por_umbral = pd.Series(
    [f1_score(validacion["etiqueta"], [clasificar_tono(tono, umbral) for tono in validacion["tono_m1"]], average="macro")
     for umbral in umbrales],
    index=umbrales, name="F1 macro")
umbral_m1 = f1_por_umbral.idxmax()   # idxmax devuelve el primero si hay empate: el umbral más pequeño

grafico = px.line(f1_por_umbral.reset_index(), x="index", y="F1 macro", markers=True,
                  labels={"index": "Umbral de neutral"}, title="M1: F1 macro en la validación según el umbral")
grafico.add_hline(y=0.248, line_dash="dash", annotation_text="siempre neutral")
grafico.show(renderer="notebook_connected")
print(f"Umbral elegido: {umbral_m1} · F1 macro en validación: {f1_por_umbral.max():.3f}")

Umbral elegido: 0.0 · F1 macro en validación: 0.509


Los umbrales de 0 a 0,02 dan exactamente lo mismo: las frases tienen unas 20–30 palabras, así que una sola palabra de tono ya da un tono de ±0,03 o más. A partir de ahí, subir el umbral solo convierte en "neutral" frases que el diccionario sí había clasificado, y el F1 cae hasta el de "siempre neutral". El mejor umbral es **0**: basta con que haya más palabras de un signo que del otro.

M1 con ese umbral, medido con la misma función que todos los modelos:

In [23]:
validacion["prediccion_m1"] = [clasificar_tono(tono, umbral_m1) for tono in validacion["tono_m1"]]
evaluacion_m1 = evaluar_clasificacion(validacion["etiqueta"], validacion["prediccion_m1"])

print(f"M1 (validación, {evaluacion_m1['frases']} frases)")
print(f"F1 macro: {evaluacion_m1['f1_macro']:.3f}  (IC 95 %: {evaluacion_m1['f1_ic_inferior']:.3f} – {evaluacion_m1['f1_ic_superior']:.3f})")
print(f"Exactitud: {evaluacion_m1['exactitud']:.3f}")
evaluacion_m1["matriz_confusion"]

M1 (validación, 967 frases)
F1 macro: 0.509  (IC 95 %: 0.472 – 0.545)
Exactitud: 0.627


predicha,negative,neutral,positive
real,,,
negative,52,67,2
neutral,46,487,41
positive,20,185,67


Se guardan el umbral y las dos listas de palabras, para que el notebook 08 aplique exactamente el mismo M1 a los titulares sin volver a leer el diccionario:

In [24]:
RUTA_M1 = CARPETA_CACHE.parent / "modelos" / "m1_diccionario.json"
RUTA_M1.parent.mkdir(parents=True, exist_ok=True)
RUTA_M1.write_text(json.dumps({
    "umbral": float(umbral_m1),
    "palabras_negativas": sorted(palabras_negativas),
    "palabras_positivas": sorted(palabras_positivas),
}, indent=1), encoding="utf-8")
print(f"Guardado: {RUTA_M1.name} - {RUTA_M1.stat().st_size / 1e3:.0f} KB")

Guardado: m1_diccionario.json - 42 KB


**Conclusión:** sin entrenar nada, M1 llega a un **F1 macro de 0,51** en la validación (IC 95 %: 0,47 – 0,55), el doble que decir siempre "neutral" (0,25). Su límite es la **cobertura**: el 76 % de las frases no tiene ninguna palabra del diccionario, y M1 las llama neutrales. Falla sobre todo en las **positivas** (solo el 25 % tiene alguna palabra positiva, y 185 de 272 acaban como neutrales): la lista positiva es corta (347 palabras) y está pensada para informes 10-K, no para noticias. En las negativas acierta más (43 % con alguna palabra negativa). Este es el listón que tienen que superar M2 y M3, que aprenden de ejemplos.

## 5. ¿Con qué palabras trabajan M2 y M3? (vocabulario común)

Una red no lee texto, lee números. Antes de M2 y M3 hay que fijar un **vocabulario**: una lista de palabras en la que cada una tiene un número. Cada frase se convierte en la lista de números de sus palabras. Se hace como en `bert_simplificado.ipynb`, con la capa `TextVectorization` de Keras:

1. **Estandarizar** (`custom_standardization`): minúsculas y sin signos de puntuación.
2. **Aprender el vocabulario** (`adapt`): las `VOCAB_SIZE` palabras más frecuentes. Las demás se convierten en `[UNK]` ("desconocida").
3. **Añadir el token `[mask]`** al final del vocabulario: es la palabra con la que M3 tapará palabras en el preentrenamiento.
4. **Cortar o rellenar** cada frase a `MAX_LEN` posiciones; el relleno es el número 0.

M2 y M3 usan **el mismo vocabulario**, para que la única diferencia entre ellos sea el preentrenamiento. Se aprende con los textos que verán los modelos al entrenar: el corpus de preentrenamiento y las frases de **entrenamiento** de PhraseBank (nunca las de validación ni las de test).

**Diferencias con los apuntes:**
- `"[mask]"` en minúscula y `special_tokens` definido. En los apuntes la variable no existe y el valor por defecto es `"[MASK]"`; la estandarización lo pasa a minúsculas, no lo encuentra en el vocabulario y lo convierte en `[UNK]`, así que se taparían palabras con "desconocida".
- `MAX_LEN = 64` en lugar de 256: aquí son frases, no críticas de cine.
- El vocabulario se aprende con el corpus y el entrenamiento de PhraseBank. En los apuntes se aprende con el entrenamiento y el **test** de IMDB, que mete información del test en el modelo.
- `keras.layers` en lugar de `tensorflow.keras.layers` (Keras 3), y el vocabulario se guarda en disco (caché).

In [25]:
MAX_LEN = 64          # palabras por frase (se corta o se rellena)
VOCAB_SIZE = 30000    # tamaño del vocabulario, como en los apuntes


def custom_standardization(input_data):
    """Minúsculas, sin <br /> y sin signos de puntuación (como en los apuntes)."""
    lowercase = tf.strings.lower(input_data)
    stripped_html = tf.strings.regex_replace(lowercase, "<br />", " ")
    return tf.strings.regex_replace(
        stripped_html, "[%s]" % re.escape("!#$%&'()*+,-./:;<=>?@\\^_`{|}~"), ""
    )


def get_vectorize_layer(texts, vocab_size, max_seq, special_tokens=["[mask]"]):
    """Crea la capa TextVectorization, aprende el vocabulario con `texts` y añade los tokens especiales al final (como en los apuntes)."""
    vectorize_layer = TextVectorization(
        max_tokens=vocab_size,
        output_mode="int",
        standardize=custom_standardization,
        output_sequence_length=max_seq,
    )
    vectorize_layer.adapt(texts)

    # Insert mask token in vocabulary
    vocab = vectorize_layer.get_vocabulary()
    vocab = vocab[2 : vocab_size - len(special_tokens)] + special_tokens
    vectorize_layer.set_vocabulary(vocab)
    return vectorize_layer

El vocabulario se aprende una vez y se guarda en `data/modelos/vocabulario.txt` (una palabra por línea, sin el relleno ni `[UNK]`, que Keras añade solo). Al volver a ejecutar se lee de disco, así M2, M3 y el notebook 08 usan exactamente los mismos números para las mismas palabras.

In [26]:
corpus = pd.read_parquet(CARPETA_CACHE / "corpus_preentrenamiento.parquet")["texto"]
frases_entrenamiento = particion.loc[particion["parte"] == "entrenamiento", "frase"]
RUTA_VOCABULARIO = CARPETA_CACHE.parent / "modelos" / "vocabulario.txt"

if RUTA_VOCABULARIO.exists():
    vectorize_layer = TextVectorization(max_tokens=VOCAB_SIZE, output_mode="int",
                                        standardize=custom_standardization, output_sequence_length=MAX_LEN)
    vectorize_layer.set_vocabulary(RUTA_VOCABULARIO.read_text(encoding="utf-8").splitlines())
    print("Vocabulario leído de disco")
else:
    vectorize_layer = get_vectorize_layer(corpus.tolist() + frases_entrenamiento.tolist(), VOCAB_SIZE, MAX_LEN)
    RUTA_VOCABULARIO.write_text("\n".join(vectorize_layer.get_vocabulary()[2:]), encoding="utf-8")
    print("Vocabulario aprendido y guardado")

# Número del token [mask], para el preentrenamiento de M3 (como en los apuntes)
mask_token_id = int(vectorize_layer(["[mask]"]).numpy()[0][0])
vocabulario = [str(palabra) for palabra in vectorize_layer.get_vocabulary()]
print(f"Palabras en el vocabulario: {len(vocabulario)} · primeras: {vocabulario[:12]} · [mask] = {mask_token_id}")

Vocabulario leído de disco
Palabras en el vocabulario: 30000 · primeras: ['', '[UNK]', 'the', 'of', 'and', 'to', 'in', 'our', 'for', 'million', 'a', 'on'] · [mask] = 29999


Así queda una frase de la validación: el texto, los números y, de vuelta, las palabras que representa cada número. Las palabras poco frecuentes (nombres de empresa, códigos) salen como `[UNK]`:

In [27]:
frase_ejemplo = validacion["frase"].iloc[0]
numeros = vectorize_layer([frase_ejemplo]).numpy()[0]
print(frase_ejemplo, "\n")
print("Números:", numeros[numeros != 0].tolist(), f"(+ {(numeros == 0).sum()} ceros de relleno)\n")
print("Palabras:", [vocabulario[numero] for numero in numeros if numero != 0])

20 October 2010 - Finnish environmental management company Lassila & Tikanoja Oyj HEL : LAT1V , or L&T , said Monday it expects its operating profit , excluding non-recurring items , for the whole 2010 to be slightly lower than in 2009 . 

Números: [348, 290, 2581, 3941, 870, 140, 82, 1, 1, 5958, 9206, 1, 15, 12666, 1523, 2305, 171, 892, 105, 48, 221, 395, 2330, 408, 8, 2, 2522, 2581, 5, 47, 1245, 98, 170, 6, 3441] (+ 29 ceros de relleno)

Palabras: ['20', 'october', '2010', 'finnish', 'environmental', 'management', 'company', '[UNK]', '[UNK]', 'oyj', 'hel', '[UNK]', 'or', 'lt', 'said', 'monday', 'it', 'expects', 'its', 'operating', 'profit', 'excluding', 'nonrecurring', 'items', 'for', 'the', 'whole', '2010', 'to', 'be', 'slightly', 'lower', 'than', 'in', '2009']


**¿Basta con 64 palabras?** Se cuentan las palabras de cada frase después de estandarizar, sin cortar:

In [28]:
def contar_palabras(textos: list[str]) -> np.ndarray:
    """Palabras de cada texto después de estandarizarlo, sin cortar a MAX_LEN."""
    return tf.strings.split(custom_standardization(textos)).row_lengths().numpy()


longitudes = {"PhraseBank": contar_palabras(particion["frase"].tolist()), "Corpus": contar_palabras(corpus.tolist())}
pd.DataFrame({
    nombre: {"mediana": np.median(n), "percentil 95": np.percentile(n, 95), "máximo": n.max(),
             f"% con más de {MAX_LEN}": round(100 * (n > MAX_LEN).mean(), 1)}
    for nombre, n in longitudes.items()
}).T

,mediana,percentil 95,máximo,% con más de 64
PhraseBank,19.0,37.0,52.0,0.0
Corpus,19.0,60.0,112.0,3.9


**¿Cuántas palabras de la validación no están en el vocabulario?** Es la parte del texto que M2 y M3 no pueden leer:

In [29]:
numeros_validacion = vectorize_layer(validacion["frase"].tolist()).numpy()
palabras_validacion = numeros_validacion[numeros_validacion != 0]
print(f"Palabras [UNK]: {(palabras_validacion == 1).mean():.1%} de las palabras de la validación")
print(f"Frases con alguna [UNK]: {(numeros_validacion == 1).any(axis=1).mean():.1%}")

Palabras [UNK]: 6.4% de las palabras de la validación
Frases con alguna [UNK]: 60.7%


**Conclusión:** vocabulario de 30.000 palabras con `[mask]` en la última posición (29.999), guardado para M2, M3 y el notebook 08. Con 64 posiciones no se corta **ninguna** frase de PhraseBank (la más larga tiene 52 palabras) y solo el 3,9 % de las del corpus, casi todas frases de informes. El 6,4 % de las palabras de la validación queda como `[UNK]`: sobre todo nombres de empresas y códigos, que no deberían decidir el sentimiento. Al quitar la puntuación, *non-recurring* pasa a ser *nonrecurring* y *L&T*, *lt*: es el precio de una estandarización sencilla como la de clase.

## 6. M2: ¿cuánto aprende un Transformer solo con las frases etiquetadas?

M2 es un **Transformer encoder** que aprende desde cero, solo con las 2.901 frases de entrenamiento de PhraseBank. Las capas son las de `Transformers.ipynb`:

- **`PositionalEmbedding`**: convierte cada número de palabra en un vector de `EMBED_DIM` valores y le suma otro vector que depende de su **posición** en la frase. Sin él, el Transformer no sabría el orden de las palabras.
- **`TransformerEncoder`**: **atención** de varias cabezas (cada palabra mira a las demás de la frase y mezcla su información según lo relacionadas que estén), y después una pequeña red densa. Cada parte lleva una conexión residual y una normalización. La **máscara** hace que la atención ignore los ceros de relleno.

La parte final que clasifica es la de `bert_simplificado.ipynb`: `GlobalMaxPooling1D` (para cada uno de los `EMBED_DIM` valores, el máximo sobre todas las palabras), una capa densa de 64 neuronas y la salida de 3 clases. **M3 tendrá exactamente la misma red**; solo cambiará que su encoder llegue preentrenado.

**Diferencias con los apuntes:**
- **3 clases:** salida `Dense(3, activation="softmax")` y pérdida `sparse_categorical_crossentropy`, en lugar de `Dense(1, activation="sigmoid")` y `binary_crossentropy`.
- **Parte final de `bert_simplificado`** (pooling + `Dense(64, relu)`) en lugar de la de `Transformers.ipynb` (pooling + `Dropout(0.5)`), y **optimizador Adam** (el de `bert_simplificado`) en lugar de *rmsprop*: así M2 y M3 solo se diferencian en el preentrenamiento.
- **Tamaño de `bert_simplificado`** (128 valores por palabra, 8 cabezas, 128 neuronas en la red densa) en lugar de 256, 2 y 32.
- `compute_mask` usa `keras.ops.not_equal` en lugar de `tf.math.not_equal`: Keras 3 la llama con un tensor simbólico al construir el modelo.
- **Las épocas se eligen con la validación**, no con el test, mediante `EarlyStopping`. Este *callback* no está en los notebooks de clase, pero sí en el libro del que salen: François Chollet, *Deep Learning with Python* (2.ª ed., capítulo 7, apartado sobre *callbacks*).
- **Pesos de clase** (se prueban con y sin ellos) y **semilla fija**, para que *Restart & Run All* dé las mismas cifras.

In [30]:
EMBED_DIM = 128   # valores por palabra (como en bert_simplificado)
NUM_HEAD = 8      # cabezas de atención
FF_DIM = 128      # neuronas de la red densa del encoder
BATCH_SIZE = 32

# Determinismo: con la misma semilla, TensorFlow da exactamente los mismos resultados en cada ejecución (algo más despacio)
tf.config.experimental.enable_op_determinism()

# Keras avisa de que TransformerEncoder "destruye la máscara": la usa dentro de la atención (llega a call como
# argumento `mask`), pero no la pasa a la capa siguiente. Es lo mismo que en los apuntes; se explica en 6.1.
warnings.filterwarnings("ignore", message=".*does not support masking.*")


class TransformerEncoder(layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.dense_dim = dense_dim
        self.num_heads = num_heads
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim)
        self.dense_proj = keras.Sequential(
            [layers.Dense(dense_dim, activation="relu"),
             layers.Dense(embed_dim),]
        )
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()

    def call(self, inputs, mask=None):  # Aquí se construye el Transformer
        if mask is not None:
            mask = mask[:, tf.newaxis, :]
        attention_output = self.attention(
            inputs, inputs, attention_mask=mask)
        proj_input = self.layernorm_1(inputs + attention_output)
        proj_output = self.dense_proj(proj_input)
        return self.layernorm_2(proj_input + proj_output)

    # sirve para serializar, lo que hace posible hacer save y load del modelo
    def get_config(self):
        config = super().get_config()
        config.update({
            "embed_dim": self.embed_dim,
            "num_heads": self.num_heads,
            "dense_dim": self.dense_dim,
        })
        return config


class PositionalEmbedding(layers.Layer):
    def __init__(self, sequence_length, input_dim, output_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_embeddings = layers.Embedding(
            input_dim=input_dim, output_dim=output_dim)
        self.position_embeddings = layers.Embedding(
            input_dim=sequence_length, output_dim=output_dim)
        self.sequence_length = sequence_length
        self.input_dim = input_dim
        self.output_dim = output_dim

    def call(self, inputs):
        length = tf.shape(inputs)[-1]
        positions = tf.range(start=0, limit=length, delta=1)
        embedded_tokens = self.token_embeddings(inputs)
        embedded_positions = self.position_embeddings(positions)
        return embedded_tokens + embedded_positions

    def compute_mask(self, inputs, mask=None):
        return keras.ops.not_equal(inputs, 0)   # Diferencia con los apuntes: tf.math.not_equal no vale en Keras 3

    def get_config(self):
        config = super().get_config()
        config.update({
            "output_dim": self.output_dim,
            "sequence_length": self.sequence_length,
            "input_dim": self.input_dim,
        })
        return config


CAPAS_PROPIAS = {"TransformerEncoder": TransformerEncoder, "PositionalEmbedding": PositionalEmbedding}   # para cargar los .keras

### 6.1 Datos y red

Cada frase pasa a sus 64 números con el vocabulario del paso 5, y cada etiqueta a un número: 0 = negativa, 1 = neutral, 2 = positiva.

In [31]:
def preparar(parte: str) -> tuple[np.ndarray, np.ndarray]:
    """Números de las frases de una parte de la partición y sus etiquetas (0, 1, 2)."""
    tabla = particion[particion["parte"] == parte]
    x = vectorize_layer(tabla["frase"].tolist()).numpy()
    y = tabla["etiqueta"].map({clase: numero for numero, clase in enumerate(CLASES_SENTIMIENTO)}).to_numpy()
    return x, y


x_entrenamiento, y_entrenamiento = preparar("entrenamiento")
x_validacion, y_validacion = preparar("validacion")
print("Entrenamiento:", x_entrenamiento.shape, "· validación:", x_validacion.shape)

Entrenamiento: (2901, 64) · validación: (967, 64)


La red de M2. Se construye con una función porque se entrena dos veces (con y sin pesos de clase) y cada vez debe empezar desde cero, con la misma semilla.

Sobre la **máscara**: `PositionalEmbedding` marca qué posiciones son relleno y Keras se lo pasa a `TransformerEncoder`, que lo usa para que la atención no mire los ceros. El encoder no pasa la máscara a la capa siguiente, así que `GlobalMaxPooling1D` también ve las posiciones de relleno. Ocurre igual en los apuntes, y Keras lo avisa (el aviso se oculta en la celda de constantes).

In [32]:
def crear_m2():
    """Transformer encoder (Transformers.ipynb) con la parte final de bert_simplificado, sin preentrenar."""
    keras.utils.set_random_seed(SEMILLA)   # mismos pesos iniciales cada vez
    inputs = keras.Input(shape=(None,), dtype="int64")
    x = PositionalEmbedding(MAX_LEN, VOCAB_SIZE, EMBED_DIM)(inputs)
    x = TransformerEncoder(EMBED_DIM, FF_DIM, NUM_HEAD)(x)
    x = layers.GlobalMaxPooling1D()(x)
    x = layers.Dense(64, activation="relu")(x)
    outputs = layers.Dense(3, activation="softmax")(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer=keras.optimizers.Adam(),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model


crear_m2().summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_embeddi… │ (None, None, 128) │  3,848,192 │ input_layer[0][0] │
│ (PositionalEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, None)      │          0 │ input_layer[0][0] │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_encoder │ (None, None, 128) │    561,024 │ positional_embed… │
│ (TransformerEncode… │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 128)       │          0 │ transformer_enco… │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │      8,256 │ global_max_pooli… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 3)         │        195 │ dense_2[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,417,667 (16.85 MB)

 Trainable params: 4,417,667 (16.85 MB)

 Non-trainable params: 0 (0.00 B)

### 6.2 Cuántas épocas: `EarlyStopping`

Se entrena hasta un máximo holgado de `MAX_EPOCAS` y, al final de cada época, se mira la **pérdida en la validación**. Si no mejora en `PACIENCIA` épocas seguidas, el entrenamiento se para, y con `restore_best_weights=True` el modelo se queda con los pesos de la **mejor época**, no con los de la última.

Durante el entrenamiento se ve la barra de progreso de Keras, época a época y lote a lote. `entrenar` guarda después el modelo y su historial en `data/modelos/`: al volver a ejecutar se leen de disco, no se entrena otra vez, y se escribe el historial guardado época a época. La misma función servirá para las dos fases del ajuste de M3.

In [33]:
MAX_EPOCAS = 50
PACIENCIA = 3
CARPETA_MODELOS = CARPETA_CACHE.parent / "modelos"


def mostrar_historial(historial: dict) -> None:
    """Escribe el historial guardado época a época, como lo escribe Keras al entrenar."""
    epocas = len(historial["loss"])
    for numero in range(epocas):
        medidas = " - ".join(f"{clave}: {valores[numero]:.4f}" for clave, valores in historial.items() if isinstance(valores, list))
        print(f"Epoch {numero + 1}/{epocas} - {medidas}")


def entrenar(crear_modelo, nombre: str, pesos_de_clase: dict | None = None):
    """Entrena con EarlyStopping sobre la pérdida de validación, o lo lee de disco si ya está entrenado.

    Al entrenar se ve la barra de progreso de Keras; al leerlo de disco se escribe el historial guardado.
    Devuelve el modelo (con los pesos de la mejor época) y su historial (pérdidas por época).
    """
    ruta_modelo = CARPETA_MODELOS / f"{nombre}.keras"
    ruta_historial = CARPETA_MODELOS / f"{nombre}_historial.json"
    if ruta_modelo.exists() and ruta_historial.exists():
        historial = json.loads(ruta_historial.read_text(encoding="utf-8"))
        print(f"{nombre}: ya entrenado, se lee de disco. Así fue su entrenamiento:")
        mostrar_historial(historial)
        return keras.models.load_model(ruta_modelo, custom_objects=CAPAS_PROPIAS), historial

    model = crear_modelo()
    parada = keras.callbacks.EarlyStopping(monitor="val_loss", patience=PACIENCIA, restore_best_weights=True)
    historia = model.fit(x_entrenamiento, y_entrenamiento, validation_data=(x_validacion, y_validacion),
                         epochs=MAX_EPOCAS, batch_size=BATCH_SIZE, class_weight=pesos_de_clase,
                         callbacks=[parada])
    historial = {clave: [float(valor) for valor in valores] for clave, valores in historia.history.items()}
    model.save(ruta_modelo)
    ruta_historial.write_text(json.dumps(historial), encoding="utf-8")
    print(f"{nombre}: {len(historial['loss'])} épocas, la mejor es la {int(np.argmin(historial['val_loss'])) + 1}")
    return model, historial


def dibujar_perdidas(historial: dict, titulo: str) -> None:
    """Pérdida de entrenamiento y de validación por época, con la época elegida marcada."""
    epocas = list(range(1, len(historial["loss"]) + 1))
    mejor = int(np.argmin(historial["val_loss"])) + 1
    grafico = px.line(pd.DataFrame({"época": epocas, "entrenamiento": historial["loss"], "validación": historial["val_loss"]}),
                      x="época", y=["entrenamiento", "validación"], markers=True,
                      labels={"value": "pérdida", "variable": ""}, title=titulo)
    grafico.add_vline(x=mejor, line_dash="dash", annotation_text=f"época elegida: {mejor}")
    grafico.show(renderer="notebook_connected")

### 6.3 Con y sin pesos de clase

Con el desequilibrio de PhraseBank (59 % neutrales, 12,5 % negativas), una red puede aprender a decir casi siempre "neutral". Los **pesos de clase** hacen que equivocarse en una frase de una clase poco frecuente cuente más en la pérdida. Con la opción `"balanced"`, el peso de cada clase es *frases totales / (3 × frases de esa clase)*:

In [34]:
pesos = compute_class_weight("balanced", classes=np.arange(3), y=y_entrenamiento)
pesos_de_clase = dict(enumerate(pesos))
print({clase: round(peso, 2) for clase, peso in zip(CLASES_SENTIMIENTO, pesos)})

{'negative': np.float64(2.67), 'neutral': np.float64(0.56), 'positive': np.float64(1.18)}


Se entrena M2 dos veces, sin y con pesos de clase, con la misma semilla y las mismas épocas máximas:

In [35]:
m2_sin_pesos, historial_sin_pesos = entrenar(crear_m2, "m2_sin_pesos")
m2_con_pesos, historial_con_pesos = entrenar(crear_m2, "m2_con_pesos", pesos_de_clase)

Epoch 1/50


E0000 00:00:1793550792.725128  276229 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


91/91 ━━━━━━━━━━━━━━━━━━━━ 5s 43ms/step - accuracy: 0.6460 - loss: 0.8226 - val_accuracy: 0.6918 - val_loss: 0.7348
Epoch 2/50
91/91 ━━━━━━━━━━━━━━━━━━━━ 4s 41ms/step - accuracy: 0.8666 - loss: 0.3649 - val_accuracy: 0.7539 - val_loss: 0.6633
Epoch 3/50
91/91 ━━━━━━━━━━━━━━━━━━━━ 4s 41ms/step - accuracy: 0.9617 - loss: 0.1109 - val_accuracy: 0.7373 - val_loss: 0.8291
Epoch 4/50
91/91 ━━━━━━━━━━━━━━━━━━━━ 4s 42ms/step - accuracy: 0.9762 - loss: 0.0723 - val_accuracy: 0.6753 - val_loss: 1.0864
Epoch 5/50
91/91 ━━━━━━━━━━━━━━━━━━━━ 4s 41ms/step - accuracy: 0.9755 - loss: 0.0661 - val_accuracy: 0.6225 - val_loss: 1.3116
m2_sin_pesos: 5 épocas, la mejor es la 2
Epoch 1/50


E0000 00:00:1793550812.625057  276229 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}


91/91 ━━━━━━━━━━━━━━━━━━━━ 5s 43ms/step - accuracy: 0.5474 - loss: 0.9767 - val_accuracy: 0.7239 - val_loss: 0.6915
Epoch 2/50
91/91 ━━━━━━━━━━━━━━━━━━━━ 4s 41ms/step - accuracy: 0.8659 - loss: 0.3756 - val_accuracy: 0.7301 - val_loss: 0.6917
Epoch 3/50
91/91 ━━━━━━━━━━━━━━━━━━━━ 4s 42ms/step - accuracy: 0.9548 - loss: 0.1241 - val_accuracy: 0.7394 - val_loss: 0.8105
Epoch 4/50
91/91 ━━━━━━━━━━━━━━━━━━━━ 4s 42ms/step - accuracy: 0.9679 - loss: 0.0878 - val_accuracy: 0.7229 - val_loss: 0.9056
m2_con_pesos: 4 épocas, la mejor es la 1


In [36]:
dibujar_perdidas(historial_sin_pesos, "M2 sin pesos de clase: pérdida por época")
dibujar_perdidas(historial_con_pesos, "M2 con pesos de clase: pérdida por época")

Las dos versiones se miden en la validación con la misma función que todos los modelos, y M2 es la que tenga **mayor F1 macro**:

In [37]:
def predecir(model, x: np.ndarray) -> list[str]:
    """Clase con más probabilidad para cada frase."""
    return [CLASES_SENTIMIENTO[numero] for numero in model.predict(x, verbose=0).argmax(axis=1)]


reales_validacion = [CLASES_SENTIMIENTO[numero] for numero in y_validacion]
evaluaciones_m2 = {
    "sin pesos de clase": evaluar_clasificacion(reales_validacion, predecir(m2_sin_pesos, x_validacion)),
    "con pesos de clase": evaluar_clasificacion(reales_validacion, predecir(m2_con_pesos, x_validacion)),
}
pd.DataFrame({
    version: {"F1 macro": e["f1_macro"], "IC 95 % inferior": e["f1_ic_inferior"], "IC 95 % superior": e["f1_ic_superior"],
              "exactitud": e["exactitud"]}
    for version, e in evaluaciones_m2.items()
}).T.round(3)

E0000 00:00:1793550828.840627  276229 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


,F1 macro,IC 95 % inferior,IC 95 % superior,exactitud
sin pesos de clase,0.675,0.639,0.712,0.754
con pesos de clase,0.621,0.582,0.658,0.724


In [38]:
version_m2 = max(evaluaciones_m2, key=lambda version: evaluaciones_m2[version]["f1_macro"])
m2 = m2_con_pesos if version_m2 == "con pesos de clase" else m2_sin_pesos
evaluacion_m2 = evaluaciones_m2[version_m2]
(CARPETA_MODELOS / "m2_eleccion.json").write_text(json.dumps({"version": version_m2}), encoding="utf-8")

print(f"M2 elegido: {version_m2}")
print(f"F1 macro: {evaluacion_m2['f1_macro']:.3f}  (IC 95 %: {evaluacion_m2['f1_ic_inferior']:.3f} – {evaluacion_m2['f1_ic_superior']:.3f})")
print(f"Exactitud: {evaluacion_m2['exactitud']:.3f}")
evaluacion_m2["matriz_confusion"]

M2 elegido: sin pesos de clase
F1 macro: 0.675  (IC 95 %: 0.639 – 0.712)
Exactitud: 0.754


predicha,negative,neutral,positive
real,,,
negative,54,42,25
neutral,8,513,53
positive,11,99,162


### 6.4 ¿Se queda con la mejor época aunque no llegue a pararse?

Si el entrenamiento llega a `MAX_EPOCAS` sin que salte la parada, ¿se restauran igualmente los pesos de la mejor época? Se comprueba repitiendo el entrenamiento de M2 elegido, con la misma semilla, pero con un máximo de **la mejor época + 2**: como la paciencia es 3, la parada no puede saltar, y la última época no es la mejor. Si los pesos se restauran, la pérdida del modelo final en la validación es la mínima del historial, no la de la última época.

La repetición sigue exactamente el mismo camino porque al principio de la sección se activó el **determinismo** de TensorFlow.

In [39]:
historial_m2 = historial_con_pesos if version_m2 == "con pesos de clase" else historial_sin_pesos
mejor_epoca = int(np.argmin(historial_m2["val_loss"])) + 1

prueba = crear_m2()
parada_prueba = keras.callbacks.EarlyStopping(monitor="val_loss", patience=PACIENCIA, restore_best_weights=True)
historia_prueba = prueba.fit(x_entrenamiento, y_entrenamiento, validation_data=(x_validacion, y_validacion),
                             epochs=mejor_epoca + 2, batch_size=BATCH_SIZE,
                             class_weight=pesos_de_clase if version_m2 == "con pesos de clase" else None,
                             callbacks=[parada_prueba]).history
perdida_final = prueba.evaluate(x_validacion, y_validacion, verbose=0)[0]

print(f"Épocas: {len(historia_prueba['val_loss'])} de {mejor_epoca + 2} · ¿saltó la parada?: {parada_prueba.stopped_epoch > 0}")
print("Pérdida de validación por época:", np.round(historia_prueba["val_loss"], 4).tolist())
print(f"Pérdida del modelo final: {perdida_final:.4f} · mínima del historial: {min(historia_prueba['val_loss']):.4f}"
      f" · última época: {historia_prueba['val_loss'][-1]:.4f}")

Epoch 1/4
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.6460 - loss: 0.8226

E0000 00:00:1793550835.737447  276229 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


91/91 ━━━━━━━━━━━━━━━━━━━━ 5s 44ms/step - accuracy: 0.6460 - loss: 0.8226 - val_accuracy: 0.6918 - val_loss: 0.7348
Epoch 2/4
91/91 ━━━━━━━━━━━━━━━━━━━━ 4s 43ms/step - accuracy: 0.8666 - loss: 0.3649 - val_accuracy: 0.7539 - val_loss: 0.6633
Epoch 3/4
91/91 ━━━━━━━━━━━━━━━━━━━━ 4s 42ms/step - accuracy: 0.9617 - loss: 0.1109 - val_accuracy: 0.7373 - val_loss: 0.8291
Epoch 4/4
91/91 ━━━━━━━━━━━━━━━━━━━━ 4s 43ms/step - accuracy: 0.9762 - loss: 0.0723 - val_accuracy: 0.6753 - val_loss: 1.0864


E0000 00:00:1793550848.010553  276229 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


Épocas: 4 de 4 · ¿saltó la parada?: False
Pérdida de validación por época: [0.7348, 0.6633, 0.8291, 1.0864]
Pérdida del modelo final: 0.6633 · mínima del historial: 0.6633 · última época: 1.0864


Funciona: el entrenamiento llega al máximo sin que salte la parada, y aun así el modelo final tiene la pérdida de la mejor época, no la de la última.

**Conclusión:** M2 **sin pesos de clase** llega en la validación a un **F1 macro de 0,675** (IC 95 %: 0,64 – 0,71), frente a 0,51 de M1: los intervalos no se solapan. Aprender de 2.901 frases etiquetadas supera a la lista de palabras de los expertos. La comparación definitiva es la del test, en el paso 9.

- **Se sobreajusta muy pronto.** La mejor época es la 2 (sin pesos) o la 1 (con pesos). Desde ahí, la pérdida de entrenamiento sigue bajando (de 0,37 a 0,07) mientras la de validación sube (de 0,66 a 1,31): la red memoriza las frases de entrenamiento. Tiene 4,4 millones de parámetros para 2.901 frases. Es justo lo que el preentrenamiento de M3 podría mejorar.
- **Los pesos de clase no ayudan aquí:** 0,621 frente a 0,675, aunque los intervalos se tocan (0,58 – 0,66 y 0,64 – 0,71). Se queda la versión sin pesos.
- **Errores:** acierta 54 de 121 negativas, 513 de 574 neutrales y 162 de 272 positivas. Las negativas siguen siendo lo más difícil: 42 acaban como neutrales y 25 como positivas.

## 7. M3, primera fase: preentrenar la red adivinando palabras tapadas

M3 tiene la misma red que M2, pero antes de ver ninguna frase etiquetada **aprende cómo es el lenguaje financiero** con el corpus propio (695.136 frases de titulares e informes, sin etiquetas). Es el preentrenamiento de **BERT**, como en `bert_simplificado.ipynb`: se tapan algunas palabras de cada frase y la red tiene que adivinarlas mirando las de alrededor. Para acertar necesita aprender qué palabras van juntas y en qué contexto, y eso es lo que después aprovechará para el sentimiento.

**Cómo se tapan las palabras** (`get_masked_input_and_labels`, copiada de los apuntes): se elige al azar el 15 % de las palabras de cada frase (nunca el relleno ni `[UNK]`). De las elegidas, el 80 % se cambia por `[mask]`, el 10 % por una palabra al azar y el 10 % se deja igual, para que la red no aprenda que solo tiene que fijarse en `[mask]`. La pérdida solo cuenta en las palabras elegidas: las demás tienen peso 0 (`sample_weights`).

**Diferencias con los apuntes:**
- **La red es la de M2** (`PositionalEmbedding` + `TransformerEncoder` de `Transformers.ipynb`) en lugar de `bert_module`, para que M3 y M2 sean la misma red. Por eso el embedding de posición se aprende, en lugar de empezar con la tabla de senos y cosenos de los apuntes, y está dentro de una capa (en los apuntes, `tf.range` va fuera de una capa, y eso no funciona en Keras 3).
- **Pérdida con la reducción por defecto** en lugar de `Reduction.NONE`: los `sample_weights` ya anulan las palabras no elegidas.
- Nombres de capa sin `/` (`encoder_0` en lugar de `encoder_0/...`), que Keras 3 no admite; `.keras` en lugar de `.h5`; semilla fija en el enmascarado y en el orden de los lotes.
- Igual que en los apuntes, la palabra número 2 del vocabulario (*the*) nunca se tapa: la condición `encoded_texts <= 2` se escribió para un vocabulario en el que los números 0, 1 y 2 eran especiales.

In [40]:
def get_masked_input_and_labels(encoded_texts):
    """Tapa el 15 % de las palabras como BERT (como en los apuntes). Devuelve frases tapadas, palabras originales y pesos."""
    # 15% BERT masking
    inp_mask = np.random.rand(*encoded_texts.shape) < 0.15
    # Do not mask special tokens
    inp_mask[encoded_texts <= 2] = False
    # Set targets to -1 by default, it means ignore
    labels = -1 * np.ones(encoded_texts.shape, dtype=int)
    # Set labels for masked tokens
    labels[inp_mask] = encoded_texts[inp_mask]

    # Prepare input
    encoded_texts_masked = np.copy(encoded_texts)
    # Set input to [MASK] which is the last token for the 90% of tokens
    # This means leaving 10% unchanged
    inp_mask_2mask = inp_mask & (np.random.rand(*encoded_texts.shape) < 0.90)
    encoded_texts_masked[inp_mask_2mask] = mask_token_id  # mask token is the last in the dict

    # Set 10% to a random token
    inp_mask_2random = inp_mask_2mask & (np.random.rand(*encoded_texts.shape) < 1 / 9)
    encoded_texts_masked[inp_mask_2random] = np.random.randint(
        3, mask_token_id, inp_mask_2random.sum()
    )

    # Prepare sample_weights to pass to .fit() method
    sample_weights = np.ones(labels.shape)
    sample_weights[labels == -1] = 0

    # y_labels would be same as encoded_texts i.e input tokens
    y_labels = np.copy(encoded_texts)

    return encoded_texts_masked, y_labels, sample_weights

Se convierte todo el corpus a números y se tapa una sola vez, con la semilla fija (en los apuntes también se tapa una sola vez, antes de entrenar):

In [41]:
x_all_review = vectorize_layer(corpus.tolist()).numpy()
np.random.seed(SEMILLA)
x_masked_train, y_masked_labels, sample_weights = get_masked_input_and_labels(x_all_review)

tapadas = sample_weights > 0
print(f"Frases: {len(x_all_review):,} · palabras tapadas: {tapadas.sum() / (x_all_review != 0).sum():.1%} de las palabras reales")
print(f"De las tapadas: {(x_masked_train[tapadas] == mask_token_id).mean():.1%} con [mask], "
      f"{(x_masked_train[tapadas] == x_all_review[tapadas]).mean():.1%} sin cambiar y el resto, una palabra al azar")

ejemplo = 0
print("\nOriginal:", " ".join(vocabulario[n] for n in x_all_review[ejemplo] if n != 0))
print("Tapada:  ", " ".join(vocabulario[n] for n in x_masked_train[ejemplo] if n != 0))

Frases: 695,136 · palabras tapadas: 14.1% de las palabras reales
De las tapadas: 80.0% con [mask], 10.0% sin cambiar y el resto, una palabra al azar

Original: 6 stocks that are beating the market
Tapada:   6 stocks that are beating the [mask]


La red de preentrenamiento: la de M2 hasta el encoder y, encima, una capa con una salida por cada palabra del vocabulario (`mlm_cls`), que da la probabilidad de cada palabra en cada posición. Es lo que en los apuntes es `create_masked_language_bert_model`:

In [42]:
def create_masked_language_bert_model():
    """Red de preentrenamiento: PositionalEmbedding + TransformerEncoder (la de M2) + una salida por palabra del vocabulario."""
    keras.utils.set_random_seed(SEMILLA)
    inputs = layers.Input((MAX_LEN,), dtype="int64")
    embeddings = PositionalEmbedding(MAX_LEN, VOCAB_SIZE, EMBED_DIM)(inputs)
    encoder_output = TransformerEncoder(EMBED_DIM, FF_DIM, NUM_HEAD, name="encoder_0")(embeddings)
    mlm_output = layers.Dense(VOCAB_SIZE, name="mlm_cls", activation="softmax")(encoder_output)
    mlm_model = keras.Model(inputs, mlm_output)
    mlm_model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                      loss=keras.losses.SparseCategoricalCrossentropy())
    return mlm_model


create_masked_language_bert_model().summary()

Model: "functional_9"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_8       │ (None, 64)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_embeddi… │ (None, 64, 128)   │  3,848,192 │ input_layer_8[0]… │
│ (PositionalEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_4         │ (None, 64)        │          0 │ input_layer_8[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_0           │ (None, 64, 128)   │    561,024 │ positional_embed… │
│ (TransformerEncode… │                   │            │ not_equal_4[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mlm_cls (Dense)     │ (None, 64, 30000) │  3,870,000 │ encoder_0[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 8,279,216 (31.58 MB)

 Trainable params: 8,279,216 (31.58 MB)

 Non-trainable params: 0 (0.00 B)

**Entrenamiento: el corpus entero, 2 épocas, como en los apuntes** (81 minutos en el Mac del autor; una prueba previa de 110 lotes estimaba unos 66). Sin validación ni `EarlyStopping`: aquí no hay etiquetas que proteger, y lo que importa es cómo funcione después M3 con PhraseBank.

Durante el entrenamiento se ve la barra de progreso de Keras (unos 21.700 lotes por época). Se entrena **una sola vez**: el modelo y su historial se guardan en `data/modelos/` y al volver a ejecutar se leen de disco y se escribe el historial época a época. Como en los apuntes, los datos se pasan con `tf.data` (barajados por lotes de 1.000, con semilla).

In [43]:
RUTA_MLM = CARPETA_MODELOS / "m3_preentrenado.keras"
RUTA_HISTORIAL_MLM = CARPETA_MODELOS / "m3_preentrenado_historial.json"
EPOCAS_PREENTRENAMIENTO = 2

if RUTA_MLM.exists() and RUTA_HISTORIAL_MLM.exists():
    bert_masked_model = keras.models.load_model(RUTA_MLM, custom_objects=CAPAS_PROPIAS)
    historial_mlm = json.loads(RUTA_HISTORIAL_MLM.read_text(encoding="utf-8"))
    print("Ya preentrenado, se lee de disco. Así fue su entrenamiento:")
    mostrar_historial(historial_mlm)
else:
    mlm_ds = tf.data.Dataset.from_tensor_slices((x_masked_train, y_masked_labels, sample_weights))
    mlm_ds = mlm_ds.shuffle(1000, seed=SEMILLA).batch(BATCH_SIZE)
    bert_masked_model = create_masked_language_bert_model()
    inicio = time.time()
    historia = bert_masked_model.fit(mlm_ds, epochs=EPOCAS_PREENTRENAMIENTO)
    historial_mlm = {"loss": [float(valor) for valor in historia.history["loss"]],
                     "minutos": round((time.time() - inicio) / 60, 1)}
    bert_masked_model.save(RUTA_MLM)
    RUTA_HISTORIAL_MLM.write_text(json.dumps(historial_mlm), encoding="utf-8")
    print("Preentrenado y guardado")

print(f"Pérdida por época: {np.round(historial_mlm['loss'], 4).tolist()} · duración: {historial_mlm['minutos']} min")

Epoch 1/2


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning:

`shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).



21723/21723 ━━━━━━━━━━━━━━━━━━━━ 2398s 110ms/step - loss: 0.2059
Epoch 2/2
21723/21723 ━━━━━━━━━━━━━━━━━━━━ 2439s 112ms/step - loss: 0.1658
Preentrenado y guardado
Pérdida por época: [0.2059, 0.1658] · duración: 80.6 min


La pérdida de Keras es la media sobre **todas** las posiciones, incluidas las que tienen peso 0, así que es un número pequeño y poco intuitivo. Para ver si la red ha aprendido algo se mide directamente con frases que **no ha visto nunca**: las de la validación de PhraseBank. Se tapan como en el preentrenamiento y se cuenta cuántas veces la palabra más probable es la correcta. Como referencia, se compara con decir siempre la palabra que más veces se tapó en el corpus. No es *the*, la más frecuente del vocabulario: como en los apuntes, los números 0 a 2 (relleno, `[UNK]` y *the*) nunca se tapan.

In [53]:
numeros_validacion_frases = vectorize_layer(validacion["frase"].tolist()).numpy()
np.random.seed(SEMILLA)
x_tapadas_val, y_tapadas_val, pesos_tapadas_val = get_masked_input_and_labels(numeros_validacion_frases)
posiciones = pesos_tapadas_val > 0

prediccion_palabras = bert_masked_model.predict(x_tapadas_val, batch_size=BATCH_SIZE, verbose=0).argmax(axis=-1)
acierto_mlm = (prediccion_palabras[posiciones] == y_tapadas_val[posiciones]).mean()
palabra_frecuente = np.bincount(y_masked_labels[sample_weights > 0]).argmax()   # la más tapada en el corpus
acierto_frecuente = (y_tapadas_val[posiciones] == palabra_frecuente).mean()
print(f"Palabras tapadas en la validación: {posiciones.sum():,}")
print(f"Acierto del modelo preentrenado: {acierto_mlm:.1%} · decir siempre '{vocabulario[palabra_frecuente]}': {acierto_frecuente:.1%}")

E0000 00:00:1793556090.495118  276229 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


Palabras tapadas en la validación: 2,537
Acierto del modelo preentrenado: 20.7% · decir siempre 'of': 3.7%


Y unos ejemplos escritos a mano, con una palabra tapada: las cinco palabras que la red ve más probables en el hueco.

In [54]:
def completar(frase: str, cuantas: int = 5) -> list[str]:
    """Las palabras más probables para el primer [mask] de una frase, según el modelo preentrenado."""
    numeros = vectorize_layer([frase]).numpy()
    posicion = int(np.where(numeros[0] == mask_token_id)[0][0])
    probabilidades = bert_masked_model.predict(numeros, verbose=0)[0, posicion]
    return [vocabulario[n] for n in probabilidades.argsort()[::-1][:cuantas]]


for frase in ["The company reported a net [mask] of EUR 5 million .",
              "Shares of the bank [mask] 10 % after the results .",
              "Operating profit rose to EUR 13.1 mn from EUR 8.7 mn in the corresponding [mask] in 2007 .",
              "We may not be able to [mask] our debt obligations ."]:
    print(f"{frase}\n   -> {completar(frase)}")

The company reported a net [mask] of EUR 5 million .
   -> ['loss', 'gain', 'charge', 'income', 'sales']
Shares of the bank [mask] 10 % after the results .
   -> ['was', 'note', 'is', 'at', 'notes']
Operating profit rose to EUR 13.1 mn from EUR 8.7 mn in the corresponding [mask] in 2007 .
   -> ['period', 'periods', 'increase', 'change', 'quarter']
We may not be able to [mask] our debt obligations .
   -> ['execute', 'maintain', 'refinance', 'access', 'secure']


E0000 00:00:1793556096.943852  276229 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


**Conclusión del preentrenamiento.** El enmascarado es el de los apuntes: se tapa el 14,1 % de las palabras reales del corpus (algo menos del 15 % porque *the* nunca se tapa) y, de las tapadas, el 80 % se cambia por `[mask]`, el 10 % se deja igual y el 10 % se cambia por una palabra al azar. En 2 épocas la pérdida baja de 0,206 a 0,166, todavía bajando al terminar. En frases que no ha visto nunca, la red acierta la palabra tapada el 20,7 % de las veces, frente al 3,7 % de decir siempre *of*: ha aprendido algo del lenguaje financiero. Los ejemplos lo confirman en parte. Completa bien el vocabulario contable (*net loss*, *corresponding period*, *refinance our debt*). En cambio, en *Shares of the bank [mask] 10 %* no propone *rose* ni *fell*, que es justo la clase de palabra que lleva el sentimiento. Si este preentrenamiento ayuda a clasificar se ve en el paso 8, comparando M3 con M2 en la misma validación.


## 8. M3, segunda fase: ¿ayuda el preentrenamiento a clasificar el sentimiento?

Ahora M3 aprende a clasificar con las mismas 2.901 frases de entrenamiento que M2. Se hace como en `bert_simplificado.ipynb`, en dos fases:

1. **Encoder congelado:** se toma la red preentrenada hasta la salida del encoder, se congela (`trainable = False`) y encima se pone la misma parte final que en M2 (`GlobalMaxPooling1D`, `Dense(64, relu)` y la salida de 3 clases). Solo aprende esa parte final: el encoder se queda como salió del preentrenamiento.
2. **Encoder descongelado:** se descongela todo y se sigue entrenando desde la fase 1, con un `Adam()` nuevo. Ahora también se ajusta el encoder a la tarea.

**M3 es el modelo de la fase 2**, como en los apuntes. La fase 1 se mide también, para ver cuánto aporta cada fase.

**Diferencias con los apuntes:**
- **Corte a la salida de la capa `encoder_0`** en lugar de `encoder_0/ffn_layernormalization`: nuestro encoder es el de `Transformers.ipynb`, cuya salida es la de su última normalización, así que es el mismo punto de la red.
- **3 clases** y `sparse_categorical_crossentropy`, como en M2.
- **Épocas con `EarlyStopping`** sobre la validación en las dos fases, en lugar de 2 épocas fijas mirando el test (como en M2, sección 6.2).
- **Sin pesos de clase:** se mantiene la configuración elegida para M2, para que la única diferencia entre M2 y M3 sea el preentrenamiento.
- Igual que en los apuntes: `Adam()` con su velocidad por defecto (0,001) en las dos fases.

### 8.1 El encoder preentrenado, congelado

Se lee de disco el modelo preentrenado en la sección 7 y se corta a la salida del encoder: es `pretrained_bert_model`, como en los apuntes.

In [ ]:
mlm_model = keras.models.load_model(RUTA_MLM, custom_objects=CAPAS_PROPIAS)
pretrained_bert_model = keras.Model(mlm_model.input, mlm_model.get_layer("encoder_0").output, name="bert_preentrenado")

# Freeze it
pretrained_bert_model.trainable = False

El clasificador, `create_classifier_bert_model` de los apuntes: la entrada pasa por el encoder preentrenado y después por la misma parte final que M2. En el resumen se ve que, con el encoder congelado, solo se entrenan los 8.451 parámetros de la parte final; los 4,4 millones del encoder no se tocan.

In [ ]:
def create_classifier_bert_model():
    """Clasificador de bert_simplificado sobre el encoder preentrenado, con la misma parte final que M2."""
    keras.utils.set_random_seed(SEMILLA)   # mismos pesos iniciales de la parte final cada vez
    inputs = layers.Input((MAX_LEN,), dtype="int64")
    sequence_output = pretrained_bert_model(inputs)
    pooled_output = layers.GlobalMaxPooling1D()(sequence_output)
    hidden_layer = layers.Dense(64, activation="relu")(pooled_output)
    outputs = layers.Dense(3, activation="softmax")(hidden_layer)
    classifer_model = keras.Model(inputs, outputs, name="classification")
    optimizer = keras.optimizers.Adam()
    classifer_model.compile(optimizer=optimizer, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return classifer_model


create_classifier_bert_model().summary()

Fase 1: se entrena con la misma función `entrenar` que M2 (`EarlyStopping`, barra de progreso y guardado en `data/modelos/`).

In [ ]:
m3_congelado, historial_congelado = entrenar(create_classifier_bert_model, "m3_congelado")
dibujar_perdidas(historial_congelado, "M3, fase 1 (encoder congelado): pérdida por época")

### 8.2 Todo descongelado

Como en los apuntes: se descongela el encoder (`trainable = True`), se compila otra vez con un `Adam()` nuevo y se sigue entrenando. Se parte del modelo de la fase 1 guardado en disco, con los pesos de su mejor época. Se lee sin compilar (`compile=False`) porque se compila justo después con el optimizador nuevo.

In [ ]:
def descongelar():
    """El clasificador de la fase 1 (leído de disco) con el encoder descongelado y un Adam nuevo, como en los apuntes."""
    keras.utils.set_random_seed(SEMILLA)
    classifer_model = keras.models.load_model(CARPETA_MODELOS / "m3_congelado.keras", custom_objects=CAPAS_PROPIAS, compile=False)
    # Unfreeze the BERT model for fine-tuning
    classifer_model.get_layer("bert_preentrenado").trainable = True
    optimizer = keras.optimizers.Adam()
    classifer_model.compile(optimizer=optimizer, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return classifer_model


m3, historial_m3 = entrenar(descongelar, "m3_descongelado")
dibujar_perdidas(historial_m3, "M3, fase 2 (encoder descongelado): pérdida por época")

### 8.3 ¿Se queda cada fase con su mejor época?

Como en M2 (sección 6.4): si los pesos de la mejor época se restauraron, la pérdida del modelo final en la validación es la mínima de su historial, no la de la última época. Se comprueba en las dos fases.

In [ ]:
for nombre, modelo, historial in [("Fase 1 (congelado)", m3_congelado, historial_congelado),
                                  ("Fase 2 (descongelado)", m3, historial_m3)]:
    perdida = modelo.evaluate(x_validacion, y_validacion, verbose=0)[0]
    print(f"{nombre}: {len(historial['val_loss'])} épocas, la mejor es la {int(np.argmin(historial['val_loss'])) + 1}"
          f" · pérdida del modelo final: {perdida:.4f} · mínima del historial: {min(historial['val_loss']):.4f}"
          f" · última época: {historial['val_loss'][-1]:.4f}")

### 8.4 M3 frente a M2 en la validación

Todos los modelos con la misma función de evaluación y las mismas 967 frases de validación. El test sigue sin mirarse: se usa una sola vez, en el paso 9.

In [ ]:
evaluacion_m3_congelado = evaluar_clasificacion(reales_validacion, predecir(m3_congelado, x_validacion))
evaluacion_m3 = evaluar_clasificacion(reales_validacion, predecir(m3, x_validacion))

pd.DataFrame({
    nombre: {"F1 macro": e["f1_macro"], "IC 95 % inferior": e["f1_ic_inferior"], "IC 95 % superior": e["f1_ic_superior"],
             "exactitud": e["exactitud"]}
    for nombre, e in [("Siempre neutral", evaluacion_neutral), ("M1 diccionario", evaluacion_m1),
                      ("M2 Transformer", evaluacion_m2), ("M3 fase 1 (congelado)", evaluacion_m3_congelado),
                      ("M3 BERT (fase 2)", evaluacion_m3)]
}).T.round(3)

In [ ]:
print(f"M3 (validación, {evaluacion_m3['frases']} frases)")
print(f"F1 macro: {evaluacion_m3['f1_macro']:.3f}  (IC 95 %: {evaluacion_m3['f1_ic_inferior']:.3f} – {evaluacion_m3['f1_ic_superior']:.3f})")
print(f"Exactitud: {evaluacion_m3['exactitud']:.3f}")
evaluacion_m3["matriz_confusion"]

**Conclusión:** *(se escribe al ver los resultados)*

## Comprobaciones

In [55]:
# Lectura: las cifras de la ficha del dataset, tres etiquetas y ninguna frase vacía
assert len(frases_50) == 4846 and len(frases_todas) == 2264
for tabla in [frases_50, frases_todas]:
    assert set(tabla["etiqueta"]) == {"negative", "neutral", "positive"}
    assert (tabla["frase"].str.len() > 0).all()

# AllAgree es un subconjunto de 50Agree con las mismas etiquetas
pares_50 = set(zip(frases_50["frase"], frases_50["etiqueta"]))
assert all(par in pares_50 for par in zip(frases_todas["frase"], frases_todas["etiqueta"]))

# La normalización iguala frases que solo se diferencian en espacios y mayúsculas
assert normalizar("Gran , the company  GREW .") == normalizar("gran, the company grew.")

# Ninguna frase etiquetada está en el corpus de preentrenamiento
assert not en_el_corpus.any()
print("PhraseBank: OK")

PhraseBank: OK


In [56]:
# Partición: tamaños 60/20/20, sin frases repetidas dentro ni entre partes, y la misma proporción de clases
tamanos = particion["parte"].value_counts()
assert len(particion) == len(frases) == 4836
assert abs(tamanos["entrenamiento"] - 0.6 * len(particion)) <= 2
assert abs(tamanos["validacion"] - 0.2 * len(particion)) <= 2 and abs(tamanos["test"] - 0.2 * len(particion)) <= 2
assert not particion["frase"].map(normalizar).duplicated().any()
assert (proporciones[["negative", "neutral", "positive"]].max() - proporciones[["negative", "neutral", "positive"]].min()).max() < 1.0  # puntos %

# Las contradictorias no están, y en_allagree coincide con AllAgree
assert not particion["frase"].isin(contradictorias).any()
assert particion["en_allagree"].sum() == frases_todas["frase"].nunique()
print("Partición: OK")

Partición: OK


In [57]:
# Evaluación: coincide con sklearn en la referencia, el intervalo contiene el valor y la matriz cuenta todas las frases
assert abs(evaluacion_neutral["f1_macro"] - f1_score(reales, siempre_neutral, average="macro")) < 1e-12
assert abs(evaluacion_neutral["exactitud"] - accuracy_score(reales, siempre_neutral)) < 1e-12
assert evaluacion_neutral["f1_ic_inferior"] <= evaluacion_neutral["f1_macro"] <= evaluacion_neutral["f1_ic_superior"]
assert evaluacion_neutral["matriz_confusion"].values.sum() == len(reales)
assert evaluacion_neutral["matriz_confusion"]["neutral"].to_dict() == reales.value_counts().to_dict()

# Un modelo perfecto saca 1 en todo, con el intervalo [1, 1]
perfecto = evaluar_clasificacion(reales, reales)
assert perfecto["f1_macro"] == perfecto["f1_ic_inferior"] == perfecto["f1_ic_superior"] == perfecto["exactitud"] == 1

# Misma semilla, mismo intervalo; y una etiqueta desconocida da error en lugar de un resultado engañoso
assert evaluar_clasificacion(reales, siempre_neutral)["f1_ic_inferior"] == evaluacion_neutral["f1_ic_inferior"]
try:
    evaluar_clasificacion(["neutral"], ["neutro"])
    raise AssertionError("Debería haber fallado con una clase desconocida")
except ValueError:
    pass
print("Evaluación: OK")

Evaluación: OK


In [58]:
# Diccionario: el fichero esperado, una fila por palabra, sin palabras vacías, y las dos listas de tono sin solaparse
assert len(diccionario_lm) == 86553 and diccionario_lm["Word"].is_unique
assert (diccionario_lm["Word"].str.len() > 0).all() and (diccionario_lm["Word"] == diccionario_lm["Word"].str.upper()).all()
assert "NULL" in set(diccionario_lm["Word"])   # leída como palabra, no como dato vacío
assert len(palabras_negativas) == 2345 and len(palabras_positivas) == 347
assert not palabras_negativas & palabras_positivas
assert "BROKEN" not in palabras_negativas   # quitada en 2020 (valor negativo): no cuenta
print("Diccionario: OK")

Diccionario: OK


In [59]:
# M1: el tono está entre -1 y 1, el umbral elegido es el de mayor F1 (el menor si empatan) y el JSON guarda lo mismo
assert validacion["tono_m1"].between(-1, 1).all()
assert umbral_m1 in umbrales and f1_por_umbral[umbral_m1] == f1_por_umbral.max()
assert umbral_m1 == min(u for u in umbrales if f1_por_umbral[u] == f1_por_umbral.max())
assert abs(evaluacion_m1["f1_macro"] - f1_por_umbral.max()) < 1e-12
assert evaluacion_m1["f1_ic_inferior"] > 0.248   # M1 supera claramente a "siempre neutral"
guardado = json.loads(RUTA_M1.read_text(encoding="utf-8"))
assert guardado["umbral"] == umbral_m1 and set(guardado["palabras_negativas"]) == palabras_negativas
# Un ejemplo hecho a mano: 1 positiva y 2 negativas en 6 palabras (la coma y el punto no cuentan) -> tono -1/6
assert tono_loughran_mcdonald("Strong sales , but losses and declines .", {"LOSSES", "DECLINES"}, {"STRONG"}) == (1 - 2) / 6
print("M1: OK")

M1: OK


In [60]:
# Vocabulario: 30.000 palabras, [mask] la última, guardado y releído igual, y ninguna frase de PhraseBank cortada
assert len(vocabulario) == VOCAB_SIZE and vocabulario[:2] == ["", "[UNK]"]
assert mask_token_id == VOCAB_SIZE - 1 and vocabulario[mask_token_id] == "[mask]"
releido = TextVectorization(max_tokens=VOCAB_SIZE, output_mode="int", standardize=custom_standardization, output_sequence_length=MAX_LEN)
releido.set_vocabulary(RUTA_VOCABULARIO.read_text(encoding="utf-8").splitlines())
assert (releido(validacion["frase"].tolist()).numpy() == numeros_validacion).all()
assert longitudes["PhraseBank"].max() <= MAX_LEN
assert not corpus.str.contains("[mask]", case=False, regex=False).any()   # el token no aparece en ningún texto real
print("Vocabulario: OK")

Vocabulario: OK


In [61]:
# M2: la parada respeta la paciencia, el modelo guardado tiene los pesos de la mejor época y la elección es la de mayor F1
for historial in [historial_sin_pesos, historial_con_pesos]:
    epocas, mejor = len(historial["val_loss"]), int(np.argmin(historial["val_loss"])) + 1
    assert epocas == MAX_EPOCAS or epocas == mejor + PACIENCIA
assert np.isclose(m2.evaluate(x_validacion, y_validacion, verbose=0)[0], min(historial_m2["val_loss"]), atol=1e-4)
assert evaluacion_m2["f1_macro"] == max(e["f1_macro"] for e in evaluaciones_m2.values())
assert json.loads((CARPETA_MODELOS / "m2_eleccion.json").read_text(encoding="utf-8"))["version"] == version_m2

# Con el máximo de épocas alcanzado y sin parada, el modelo se queda igualmente con la mejor época (6.4)
assert parada_prueba.stopped_epoch == 0 and len(historia_prueba["val_loss"]) == mejor_epoca + 2
assert int(np.argmin(historia_prueba["val_loss"])) != len(historia_prueba["val_loss"]) - 1   # la mejor no es la última
assert np.isclose(perdida_final, min(historia_prueba["val_loss"]), atol=1e-4)
print("M2: OK")

M2: OK


In [62]:
# Preentrenamiento de M3: el enmascarado de los apuntes, 2 épocas con la pérdida bajando, y mejor que decir siempre la palabra más tapada
assert not tapadas[x_all_review <= 2].any()                                   # relleno, [UNK] y "the" nunca se tapan
assert abs(tapadas.sum() / (x_all_review > 2).sum() - 0.15) < 0.005           # el 15 % de las palabras que se pueden tapar
assert abs((x_masked_train[tapadas] == mask_token_id).mean() - 0.80) < 0.01   # 80 % con [mask]
assert abs((x_masked_train[tapadas] == x_all_review[tapadas]).mean() - 0.10) < 0.01   # 10 % sin cambiar
assert len(historial_mlm["loss"]) == EPOCAS_PREENTRENAMIENTO
assert historial_mlm["loss"][-1] < historial_mlm["loss"][0]
assert acierto_mlm > acierto_frecuente
print("Preentrenamiento de M3: OK")


Preentrenamiento de M3: OK


In [ ]:
# M3: la fase 1 deja el encoder como salió del preentrenamiento, la fase 2 lo cambia, y las dos se quedan con su mejor época
pesos_preentrenados = mlm_model.get_layer("encoder_0").get_weights()
pesos_fase_1 = m3_congelado.get_layer("bert_preentrenado").get_layer("encoder_0").get_weights()
pesos_fase_2 = m3.get_layer("bert_preentrenado").get_layer("encoder_0").get_weights()
assert all(np.array_equal(antes, despues) for antes, despues in zip(pesos_preentrenados, pesos_fase_1))
assert not all(np.array_equal(antes, despues) for antes, despues in zip(pesos_preentrenados, pesos_fase_2))

for modelo, historial in [(m3_congelado, historial_congelado), (m3, historial_m3)]:
    epocas, mejor = len(historial["val_loss"]), int(np.argmin(historial["val_loss"])) + 1
    assert epocas == MAX_EPOCAS or epocas == mejor + PACIENCIA
    assert np.isclose(modelo.evaluate(x_validacion, y_validacion, verbose=0)[0], min(historial["val_loss"]), atol=1e-4)
print("M3: OK")